# Tournament Prediction 2026 (Practice)

Practice notebook for the **group-stage** part of the 2026 World Cup pipeline. It trains goal-prediction models on historical results, applies them to the scraped 2026 schedule, and builds a simple standings table.

**Full knockout simulation** (Round of 32 through the final) lives in `Fifa_worldcup2026_TournamentPrediction.ipynb`.

## Outputs

| File | Description |
|------|-------------|
| `Data/fifa_worldcup_2026_predictions.csv` | Every 2026 fixture with predicted goals and winner |
| `Data/fifa_worldcup_2026_standings.csv` | Teams ranked by points (and goal difference) |

Section **4** at the end displays these results as tables in the notebook.

## Prerequisites

Run from the **project root** after refreshing data:

```bash
python scripts/scrape_historical_matches.py
python scripts/scrape_fixtures_2026.py
```

## 1. Train models on historical matches (1930–2022)

Two **Random Forest regressors** predict home and away goals from encoded team IDs. Teams only seen in 2026 fixtures (or knockout placeholders like `Winner Match 101`) are encoded as `-1`.

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [13]:
df = pd.read_csv("Data/clean_fifa_worldcup_matches.csv")
df = df.dropna()


In [14]:
team_encoder = LabelEncoder()
df['HomeTeamEncoded'] = team_encoder.fit_transform(df['HomeTeam'])
df['AwayTeamEncoded'] = df['AwayTeam'].apply(lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1)


In [15]:
X = df[['HomeTeamEncoded', 'AwayTeamEncoded']]
y_home = df['HomeGoals']
y_away = df['AwayGoals']

In [16]:
X_train, X_test, y_train_home, y_test_home = train_test_split(X, y_home, test_size=0.2, random_state=42)
X_train, X_test, y_train_away, y_test_away = train_test_split(X, y_away, test_size=0.2, random_state=42)


In [17]:
home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)

home_model.fit(X_train, y_train_home)
away_model.fit(X_train, y_train_away)

y_pred_home = home_model.predict(X_test)
y_pred_away = away_model.predict(X_test)

home_mae = mean_absolute_error(y_test_home, y_pred_home)
away_mae = mean_absolute_error(y_test_away, y_pred_away)

print(f"Holdout MAE — home goals: {home_mae:.3f}, away goals: {away_mae:.3f}")

Holdout MAE — home goals: 1.330, away goals: 0.935


## 2. Predict 2026 fixtures

Loads `Data/clean_fifa_worldcup_fixture.csv` (104 matches from `scripts/scrape_fixtures_2026.py`). Compares predicted home/away goals to assign a winner per match.

In [18]:
fixtures = pd.read_csv("Data/clean_fifa_worldcup_fixture.csv")

# Debugging: Print column names before renaming
print("Columns before renaming:", list(fixtures.columns))

# Rename columns correctly (matching lowercase)
fixtures.rename(columns={'home': 'HomeTeam', 'away': 'AwayTeam'}, inplace=True)

# Debugging: Print column names after renaming
print("Columns after renaming:", list(fixtures.columns))

# Check if required columns exist
if 'HomeTeam' not in fixtures.columns or 'AwayTeam' not in fixtures.columns:
    raise KeyError("Columns 'HomeTeam' or 'AwayTeam' not found after renaming!")

# Fix FutureWarning: Fill missing values correctly
fixtures['HomeTeam'] = fixtures['HomeTeam'].fillna('Unknown')
fixtures['AwayTeam'] = fixtures['AwayTeam'].fillna('Unknown')

# Encode team names
fixtures['HomeTeamEncoded'] = fixtures['HomeTeam'].apply(lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1)
fixtures['AwayTeamEncoded'] = fixtures['AwayTeam'].apply(lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1)

# Predict goals (round to integers before winner — floats rarely tie exactly)
fixtures['PredictedHomeGoals'] = home_model.predict(
    fixtures[['HomeTeamEncoded', 'AwayTeamEncoded']]
).round().astype(int)
fixtures['PredictedAwayGoals'] = away_model.predict(
    fixtures[['HomeTeamEncoded', 'AwayTeamEncoded']]
).round().astype(int)

fixtures['Winner'] = np.select(
    [
        fixtures['PredictedHomeGoals'] > fixtures['PredictedAwayGoals'],
        fixtures['PredictedHomeGoals'] < fixtures['PredictedAwayGoals'],
    ],
    [fixtures['HomeTeam'], fixtures['AwayTeam']],
    default='Draw',
)


Columns before renaming: ['home', 'score', 'away', 'year']
Columns after renaming: ['HomeTeam', 'score', 'AwayTeam', 'year']


In [19]:
fixtures.to_csv("Data/fifa_worldcup_2026_predictions.csv", index=False)


## 3. Build group standings

Aggregates **points** from predicted winners (3 / 1 / 0). This is a simplified global table—not per-group (A–L) and not using `Data/group_standings_2026.pkl` from `Table_Extraction.ipynb`.

In [20]:
group_stage_results = pd.read_csv("Data/fifa_worldcup_2026_predictions.csv")

standings = {}
for _, row in group_stage_results.iterrows():
    home, away = row['HomeTeam'], row['AwayTeam']
    home_goals = int(row['PredictedHomeGoals'])
    away_goals = int(row['PredictedAwayGoals'])

    standings.setdefault(home, {'Points': 0, 'GD': 0})
    standings.setdefault(away, {'Points': 0, 'GD': 0})

    standings[home]['GD'] += home_goals - away_goals
    standings[away]['GD'] += away_goals - home_goals

    if home_goals > away_goals:
        standings[home]['Points'] += 3
    elif away_goals > home_goals:
        standings[away]['Points'] += 3
    else:
        standings[home]['Points'] += 1
        standings[away]['Points'] += 1

standings_df = pd.DataFrame.from_dict(standings, orient='index').reset_index().rename(columns={'index': 'Team'})
standings_df = standings_df.sort_values(by=['Points', 'GD'], ascending=False)



In [21]:
standings_df.to_csv("Data/fifa_worldcup_2026_standings.csv", index=False)

print("Predictions for World Cup 2026 saved successfully!")

Predictions for World Cup 2026 saved successfully!


## Notes & limitations

- **Goal difference (`GD`)** in standings comes from summed predicted scorelines across all fixtures (including knockout placeholders).
- **Knockout placeholders** (`Winner Match 101`, etc.) are still sent through the model; many map to encoder `-1`.
- **48-team format**: this notebook does not apply FIFA’s full qualification rules (top two per group plus best third-placed teams, etc.).
- For saved models and richer score outputs, see `Prediction 2026 fifa Scores.ipynb`.

## 4. Predicted results

Summary tables below use the in-memory predictions and standings from the cells above (same data written to `Data/`).

In [22]:
from IPython.display import display

# --- Match predictions (all 2026 fixtures) ---
match_results = group_stage_results.copy()
match_results["PredictedScore"] = (
    match_results["PredictedHomeGoals"].astype(str)
    + " – "
    + match_results["PredictedAwayGoals"].astype(str)
)

matches_table = match_results.rename(columns={"score": "Match"})[
    ["Match", "HomeTeam", "PredictedScore", "AwayTeam", "Winner"]
]

print(f"Predicted scores for {len(matches_table)} fixtures:\n")
display(matches_table.style.hide(axis="index").set_properties(**{"text-align": "left"}))

# --- Standings (top 24 by points) ---
print("\nTop 24 teams by predicted group-stage points:\n")
standings_table = standings_df.head(24).copy()
display(
    standings_table.style.hide(axis="index")
    .format({"Points": "{:.0f}", "GD": "{:.0f}"})
    .set_properties(**{"text-align": "left"})
)

Predicted scores for 104 fixtures:



Match,HomeTeam,PredictedScore,AwayTeam,Winner
Match 1,Mexico,1 – 1,South Africa,Draw
Match 2,South Korea,3 – 1,Czech Republic,South Korea
Match 25,Czech Republic,1 – 1,South Africa,Draw
Match 28,Mexico,2 – 0,South Korea,Mexico
Match 53,Czech Republic,1 – 2,Mexico,Mexico
Match 54,South Africa,1 – 1,South Korea,Draw
Match 3,Canada,1 – 2,Bosnia and Herzegovina,Bosnia and Herzegovina
Match 8,Qatar,3 – 2,Switzerland,Qatar
Match 26,Switzerland,1 – 1,Bosnia and Herzegovina,Draw
Match 27,Canada,2 – 2,Qatar,Draw



Top 24 teams by predicted group-stage points:



Team,Points,GD
Brazil,7,4
Portugal,7,4
Mexico,7,3
Morocco,7,3
Sweden,7,3
Spain,7,3
Uruguay,7,3
Bosnia and Herzegovina,7,2
Ivory Coast,7,2
New Zealand,7,2
